# 02. 혼인 경험 타깃 생성과 누수 변수 차단

## tl;dr

- 32,561명 중 혼인 경험 없음은 **10,683명(32.81%)**, 혼인 경험 있음은 **21,878명(67.19%)**입니다.
- `marital-status`는 타깃 생성 후 제거하고, `relationship`은 정답을 직접 암시하므로 모델 입력에서 제외합니다.
- 기본 모델은 7개 특성, 확장 모델은 `income`을 추가한 8개 특성을 사용합니다.

## Context & Methods

### Key Assumptions

`Never-married`만 0으로 정의하고, 혼인·이혼·별거·사별 상태는 모두 1로 정의합니다. 이는 미래 결혼 가능성이 아니라 조사 시점까지의 누적 혼인 경험입니다.

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if not (project_root / 'src').exists():
    raise RuntimeError('adult-marriage-analysis 프로젝트 루트에서 실행하세요.')
sys.path.insert(0, str(project_root))

from src.data import (
    add_target_pandas,
    add_target_polars,
    compare_target_distribution,
    ensure_raw_data,
    load_with_pandas,
    load_with_polars,
)
from src.features import (
    BASE_FEATURES,
    EXTENDED_FEATURES,
    build_feature_missingness,
    build_feature_policy,
    select_feature_sets,
)

## Data

### 1. 정제 데이터에 타깃 추가

In [2]:
raw_path = ensure_raw_data(project_root / 'data' / 'raw')
pandas_prepared = add_target_pandas(load_with_pandas(raw_path))
polars_prepared = add_target_polars(load_with_polars(raw_path))

pandas_prepared[['marital-status', 'ever_married']].head()

,marital-status,ever_married
0,Never-married,0
1,Married-civ-spouse,1
2,Divorced,1
3,Married-civ-spouse,1
4,Married-civ-spouse,1


## Results

### 2. 타깃 분포와 라이브러리 간 일치

In [3]:
target_distribution = compare_target_distribution(
    pandas_prepared, polars_prepared
)
target_distribution

,ever_married,label,pandas_count,polars_count,count_equal,percentage
0,0,혼인 경험 없음,10683,10683,True,32.809189
1,1,혼인 경험 있음,21878,21878,True,67.190811


### 3. 원래 혼인 상태가 어떻게 매핑됐는지 확인

In [4]:
status_mapping_check = (
    pandas_prepared
    .groupby(['marital-status', 'ever_married'], as_index=False)
    .size()
    .sort_values(['ever_married', 'size'], ascending=[True, False])
)
status_mapping_check

,marital-status,ever_married,size
4,Never-married,0,10683
2,Married-civ-spouse,1,14976
0,Divorced,1,4443
5,Separated,1,1025
6,Widowed,1,993
3,Married-spouse-absent,1,418
1,Married-AF-spouse,1,23


### 4. 모델 입력·제외 정책 확인

In [5]:
feature_policy = build_feature_policy()
feature_policy

,column,role,reason
0,ever_married,target,marital-status에서 생성한 혼인 경험 여부
1,marital-status,exclude_target_source,타깃 생성에 사용되어 입력 시 정답 누수
2,relationship,exclude_leakage,"Husband, Wife, Unmarried가 혼인 상태를 직접 암시"
3,fnlwgt,exclude_weight,개인 특성이 아닌 Census 표본 가중치
4,education,exclude_duplicate,education-num과 같은 정보를 중복 표현
5,age,base_feature,기본 예측 모델 입력
6,education-num,base_feature,기본 예측 모델 입력
7,hours-per-week,base_feature,기본 예측 모델 입력
8,capital-gain,base_feature,기본 예측 모델 입력
9,capital-loss,base_feature,기본 예측 모델 입력


### 5. 기본·확장·공정성 데이터 분리

In [6]:
base_x, extended_x, audit_x, target = select_feature_sets(
    pandas_prepared
)

feature_set_summary = pd.DataFrame([
    {'set': '기본 모델', 'rows': len(base_x), 'columns': base_x.shape[1], 'features': ', '.join(base_x.columns)},
    {'set': '확장 모델', 'rows': len(extended_x), 'columns': extended_x.shape[1], 'features': ', '.join(extended_x.columns)},
    {'set': '공정성 점검', 'rows': len(audit_x), 'columns': audit_x.shape[1], 'features': ', '.join(audit_x.columns)},
])
feature_set_summary

,set,rows,columns,features
0,기본 모델,32561,7,"age, education-num, hours-per-week, capital-ga..."
1,확장 모델,32561,8,"age, education-num, hours-per-week, capital-ga..."
2,공정성 점검,32561,3,"race, sex, native-country"


### 6. 입력 후보의 결측치 확인

In [7]:
feature_missingness = build_feature_missingness(pandas_prepared)
feature_missingness

,column,missing_count,missing_rate_pct
0,age,0,0.000000
1,education-num,0,0.000000
2,hours-per-week,0,0.000000
3,capital-gain,0,0.000000
4,capital-loss,0,0.000000
5,workclass,1836,5.638647
6,occupation,1843,5.660146
7,income,0,0.000000


## Checks

타깃과 누수 차단 규칙을 자동 검증합니다.

In [8]:
assert set(target.unique()) == {0, 1}
assert target.notna().all()
assert target_distribution['count_equal'].all()
assert target_distribution['pandas_count'].sum() == 32_561
assert 'marital-status' not in base_x.columns
assert 'relationship' not in base_x.columns
assert 'fnlwgt' not in base_x.columns
assert 'income' not in base_x.columns
assert 'income' in extended_x.columns

print('검증 완료: 타깃 생성과 누수 변수 차단이 올바릅니다.')

검증 완료: 타깃 생성과 누수 변수 차단이 올바릅니다.


## Takeaways

1. 양성 클래스가 67.19%로 더 많으므로 단순 정확도만으로 평가하지 않습니다.
2. `workclass`와 `occupation`의 결측치는 이후 Pipeline에서 `Unknown`으로 대치합니다.
3. `income`은 현재 시점 정보이므로 기본 모델과 분리해 추가 효과만 점검합니다.
4. 다음 단계에서는 train/test를 먼저 나눈 뒤 훈련 데이터에서 EDA와 통계분석을 수행합니다.